# 📊 Análisis Cuantitativo de Vulnerabilidades y Dependencias

**Curso:** Ciberseguridad (ICC610) - 2026  
**Objetivo:** Analizar cuantitativamente las dependencias (SBOM) y vulnerabilidades (Grype/CodeQL) encontradas en repositorios open source.

## Requisitos previos
Antes de ejecutar este notebook, asegúrate de haber corrido:
```bash
uv run python main.py clone   # Clonar repositorios
uv run python main.py sbom    # Generar SBOMs
uv run python main.py grype   # Escanear vulnerabilidades
uv run python main.py codeql  # Análisis estático (opcional)
```

---

## 1. Configuración e Imports

In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter
from datetime import datetime

# Directorio de resultados
RESULTS_DIR = Path("../data/results")

# Verificar que existen resultados
sbom_files = sorted(RESULTS_DIR.glob("*-sbom.json"))
grype_files = sorted(RESULTS_DIR.glob("*-grype.json"))
codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))

print("="*60)
print("ARCHIVOS DE RESULTADOS ENCONTRADOS")
print("="*60)
print(f"📦 SBOMs:           {len(sbom_files)} archivo(s)")
print(f"🔓 Grype (vulns):   {len(grype_files)} archivo(s)")
print(f"🔍 CodeQL:          {len(codeql_files)} archivo(s)")
print("="*60)

if not sbom_files and not grype_files:
    print("\n⚠️  No se encontraron resultados. Ejecuta los análisis primero.")

---
## 2. Análisis de Dependencias (SBOM)

Analizamos los componentes de software detectados por Syft en cada repositorio.

In [ ]:
# Cargar todos los SBOMs y extraer dependencias
all_dependencies = []

for sbom_file in sbom_files:
    repo_name = sbom_file.stem.replace("-sbom", "")
    
    with open(sbom_file) as f:
        data = json.load(f)
    
    artifacts = data.get("artifacts", [])
    
    for art in artifacts:
        all_dependencies.append({
            "repo": repo_name,
            "name": art.get("name", "N/A"),
            "version": art.get("version", "N/A"),
            "type": art.get("type", "N/A"),
            "language": art.get("language", "N/A"),
            "licenses": ", ".join(
                [lic.get("value", "N/A") for lic in art.get("licenses", [])]
            ) or "No especificada",
        })

df_deps = pd.DataFrame(all_dependencies)

if not df_deps.empty:
    print(f"\n📦 Total de dependencias detectadas: {len(df_deps)}")
    print(f"📁 Repositorios analizados: {df_deps['repo'].nunique()}")
    print(f"\n--- Primeras 10 dependencias ---")
    display(df_deps.head(10))
else:
    print("⚠️  No se encontraron dependencias en los SBOMs.")

### 2.1 Dependencias por tipo de paquete

In [ ]:
if not df_deps.empty:
    print("\n📊 Distribución de dependencias por tipo de paquete:\n")
    type_counts = df_deps["type"].value_counts()
    
    for pkg_type, count in type_counts.items():
        bar = "█" * min(count, 50)
        print(f"  {pkg_type:20} │ {bar} {count}")
    
    print(f"\n  {'TOTAL':20} │ {len(df_deps)}")

### 2.2 Dependencias por repositorio

In [ ]:
if not df_deps.empty:
    print("\n📊 Cantidad de dependencias por repositorio:\n")
    repo_counts = df_deps.groupby("repo").size().sort_values(ascending=False)
    
    for repo, count in repo_counts.items():
        bar = "█" * min(count // 2, 50)
        print(f"  {repo:30} │ {bar} {count}")
    
    print(f"\n  Promedio por repositorio: {len(df_deps) / df_deps['repo'].nunique():.1f}")

### 2.3 Licencias más comunes

In [ ]:
if not df_deps.empty:
    print("\n📜 Top 10 licencias más comunes:\n")
    license_counts = df_deps["licenses"].value_counts().head(10)
    
    for lic, count in license_counts.items():
        pct = count / len(df_deps) * 100
        bar = "█" * int(pct)
        print(f"  {lic:35} │ {bar} {count} ({pct:.1f}%)")

### 2.4 Tabla resumen de dependencias

In [ ]:
if not df_deps.empty:
    summary_deps = df_deps.groupby("repo").agg(
        total_deps=("name", "count"),
        tipos_unicos=("type", "nunique"),
        paquetes_unicos=("name", "nunique"),
    ).reset_index()
    
    print("\n📋 Resumen de dependencias por repositorio:\n")
    display(summary_deps)

---
## 3. Análisis de Vulnerabilidades (Grype)

Analizamos las vulnerabilidades detectadas en las dependencias de cada repositorio.

In [ ]:
# Cargar todos los resultados de Grype
all_vulns = []

for grype_file in grype_files:
    repo_name = grype_file.stem.replace("-grype", "")
    
    with open(grype_file) as f:
        data = json.load(f)
    
    matches = data.get("matches", [])
    
    for match in matches:
        vuln = match.get("vulnerability", {})
        artifact = match.get("artifact", {})
        related = vuln.get("relatedVulnerabilities", [])
        
        # Obtener URLs de referencia
        urls = vuln.get("urls", [])
        
        all_vulns.append({
            "repo": repo_name,
            "vuln_id": vuln.get("id", "N/A"),
            "severity": vuln.get("severity", "Unknown"),
            "description": vuln.get("description", "N/A")[:100],
            "package": artifact.get("name", "N/A"),
            "version": artifact.get("version", "N/A"),
            "pkg_type": artifact.get("type", "N/A"),
            "fix_state": vuln.get("fix", {}).get("state", "N/A"),
            "fix_versions": ", ".join(vuln.get("fix", {}).get("versions", [])),
            "data_source": vuln.get("dataSource", "N/A"),
        })

df_vulns = pd.DataFrame(all_vulns)

if not df_vulns.empty:
    print(f"\n🔓 Total de vulnerabilidades detectadas: {len(df_vulns)}")
    print(f"📁 Repositorios con vulnerabilidades: {df_vulns['repo'].nunique()}")
    print(f"📦 Paquetes afectados: {df_vulns['package'].nunique()}")
    print(f"\n--- Primeras 10 vulnerabilidades ---")
    display(df_vulns[["repo", "vuln_id", "severity", "package", "version", "fix_state"]].head(10))
else:
    print("⚠️  No se encontraron vulnerabilidades (o no se ejecutó Grype).")

### 3.1 Distribución de vulnerabilidades por severidad

In [ ]:
if not df_vulns.empty:
    print("\n🎯 Distribución por severidad:\n")
    
    severity_order = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]
    severity_colors = {
        "Critical": "🔴",
        "High": "🟠",
        "Medium": "🟡",
        "Low": "🟢",
        "Negligible": "⚪",
        "Unknown": "❓"
    }
    
    sev_counts = df_vulns["severity"].value_counts()
    
    for sev in severity_order:
        if sev in sev_counts.index:
            count = sev_counts[sev]
            pct = count / len(df_vulns) * 100
            icon = severity_colors.get(sev, "")
            bar = "█" * int(pct)
            print(f"  {icon} {sev:12} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  Total: {len(df_vulns)} vulnerabilidades")

### 3.2 Vulnerabilidades por repositorio

In [ ]:
if not df_vulns.empty:
    print("\n📊 Vulnerabilidades por repositorio y severidad:\n")
    
    pivot = df_vulns.pivot_table(
        index="repo",
        columns="severity",
        values="vuln_id",
        aggfunc="count",
        fill_value=0
    )
    
    # Reordenar columnas
    cols = [c for c in severity_order if c in pivot.columns]
    pivot = pivot[cols]
    pivot["TOTAL"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("TOTAL", ascending=False)
    
    display(pivot)

### 3.3 Paquetes más vulnerables (Top 15)

In [ ]:
if not df_vulns.empty:
    print("\n📦 Top 15 paquetes con más vulnerabilidades:\n")
    
    pkg_vulns = df_vulns.groupby(["package", "version"]).agg(
        total_vulns=("vuln_id", "count"),
        critical=("severity", lambda x: (x == "Critical").sum()),
        high=("severity", lambda x: (x == "High").sum()),
        medium=("severity", lambda x: (x == "Medium").sum()),
        low=("severity", lambda x: (x == "Low").sum()),
    ).reset_index().sort_values("total_vulns", ascending=False).head(15)
    
    display(pkg_vulns)

### 3.4 Estado de las correcciones (fix available?)

In [ ]:
if not df_vulns.empty:
    print("\n🔧 Estado de correcciones disponibles:\n")
    
    fix_counts = df_vulns["fix_state"].value_counts()
    
    fix_icons = {
        "fixed": "✅",
        "not-fixed": "❌",
        "wont-fix": "🚫",
        "unknown": "❓",
        "N/A": "❓"
    }
    
    for state, count in fix_counts.items():
        pct = count / len(df_vulns) * 100
        icon = fix_icons.get(state, "")
        print(f"  {icon} {state:15} │ {count:4} ({pct:.1f}%)")
    
    # Vulnerabilidades críticas/altas sin fix
    critical_no_fix = df_vulns[
        (df_vulns["severity"].isin(["Critical", "High"])) &
        (df_vulns["fix_state"] != "fixed")
    ]
    
    if not critical_no_fix.empty:
        print(f"\n  ⚠️  Vulnerabilidades Critical/High SIN fix disponible: {len(critical_no_fix)}")

---
## 4. Análisis de Código Fuente (CodeQL)

Resultados del análisis estático del código fuente (si se ejecutó CodeQL).

In [ ]:
# Cargar resultados de CodeQL
all_codeql = []

for codeql_file in codeql_files:
    repo_name = codeql_file.stem.replace("-codeql", "")
    
    with open(codeql_file) as f:
        data = json.load(f)
    
    # CodeQL puede retornar diferentes formatos
    results = data if isinstance(data, list) else data.get("runs", [{}])[0].get("results", [])
    
    for result in results:
        if isinstance(result, dict):
            rule_id = result.get("ruleId", result.get("rule", {}).get("id", "N/A"))
            msg = result.get("message", {})
            message = msg.get("text", str(msg)) if isinstance(msg, dict) else str(msg)
            level = result.get("level", "warning")
            
            locations = result.get("locations", [{}])
            file_path = "N/A"
            line = 0
            if locations:
                phys = locations[0].get("physicalLocation", {})
                file_path = phys.get("artifactLocation", {}).get("uri", "N/A")
                line = phys.get("region", {}).get("startLine", 0)
            
            all_codeql.append({
                "repo": repo_name,
                "rule_id": rule_id,
                "level": level,
                "message": message[:120],
                "file": file_path,
                "line": line,
            })

df_codeql = pd.DataFrame(all_codeql)

if not df_codeql.empty:
    print(f"\n🔍 Total de hallazgos CodeQL: {len(df_codeql)}")
    print(f"📁 Repositorios analizados: {df_codeql['repo'].nunique()}")
    print(f"\n--- Primeros 10 hallazgos ---")
    display(df_codeql.head(10))
else:
    print("ℹ️  No se encontraron resultados de CodeQL.")
    print("   Esto es normal si no ejecutaste: uv run python main.py codeql")

In [ ]:
if not df_codeql.empty:
    print("\n📊 Hallazgos por nivel de severidad:\n")
    
    level_counts = df_codeql["level"].value_counts()
    for level, count in level_counts.items():
        pct = count / len(df_codeql) * 100
        bar = "█" * int(pct)
        print(f"  {level:15} │ {bar} {count} ({pct:.1f}%)")
    
    print("\n📋 Top 10 reglas más frecuentes:\n")
    rule_counts = df_codeql["rule_id"].value_counts().head(10)
    for rule, count in rule_counts.items():
        print(f"  {rule:50} │ {count}")

---
## 5. Resumen Ejecutivo

Consolidación de todos los hallazgos en métricas clave.

In [ ]:
print("\n" + "═"*60)
print("        📊 RESUMEN EJECUTIVO DE ANÁLISIS DE SEGURIDAD")
print("═"*60)
print(f"  Fecha del análisis:    {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Repositorios analizados: {len(sbom_files)}")
print()

# --- Dependencias ---
print("  ─── DEPENDENCIAS (SBOM) ──────────────────────")
if not df_deps.empty:
    print(f"  Total de dependencias:       {len(df_deps)}")
    print(f"  Paquetes únicos:             {df_deps['name'].nunique()}")
    print(f"  Tipos de paquete:            {df_deps['type'].nunique()}")
    print(f"  Promedio por repositorio:    {len(df_deps) / max(df_deps['repo'].nunique(), 1):.0f}")
else:
    print("  Sin datos de SBOM")

print()

# --- Vulnerabilidades ---
print("  ─── VULNERABILIDADES (Grype) ─────────────────")
if not df_vulns.empty:
    print(f"  Total de vulnerabilidades:   {len(df_vulns)}")
    for sev in ["Critical", "High", "Medium", "Low"]:
        count = len(df_vulns[df_vulns['severity'] == sev])
        icon = severity_colors.get(sev, '')
        print(f"    {icon} {sev}:{'':>{12-len(sev)}}  {count}")
    
    fixed = len(df_vulns[df_vulns['fix_state'] == 'fixed'])
    print(f"  Con fix disponible:          {fixed} ({fixed/len(df_vulns)*100:.1f}%)")
    print(f"  Paquetes afectados:          {df_vulns['package'].nunique()}")
else:
    print("  Sin datos de Grype")

print()

# --- CodeQL ---
print("  ─── ANÁLISIS ESTÁTICO (CodeQL) ───────────────")
if not df_codeql.empty:
    print(f"  Total de hallazgos:          {len(df_codeql)}")
    print(f"  Reglas únicas activadas:     {df_codeql['rule_id'].nunique()}")
else:
    print("  Sin datos de CodeQL")

print()

# --- Métricas de riesgo ---
print("  ─── MÉTRICAS DE RIESGO ───────────────────────")
if not df_vulns.empty and not df_deps.empty:
    ratio = len(df_vulns) / max(len(df_deps), 1)
    critical_high = len(df_vulns[df_vulns['severity'].isin(['Critical', 'High'])])
    print(f"  Ratio vulns/dependencias:    {ratio:.2f}")
    print(f"  Vulns Critical+High:         {critical_high}")
    
    if critical_high == 0:
        print(f"  Estado general:              🟢 BAJO RIESGO")
    elif critical_high <= 5:
        print(f"  Estado general:              🟡 RIESGO MODERADO")
    else:
        print(f"  Estado general:              🔴 ALTO RIESGO")

print("\n" + "═"*60)

---
## 6. Exportar Datos para Reportes

Exportamos los DataFrames a CSV para uso externo (Excel, Google Sheets, etc.).

In [ ]:
export_dir = RESULTS_DIR / "exports"
export_dir.mkdir(exist_ok=True)

if not df_deps.empty:
    df_deps.to_csv(export_dir / "dependencias.csv", index=False)
    print(f"✅ Dependencias exportadas a: {export_dir / 'dependencias.csv'}")

if not df_vulns.empty:
    df_vulns.to_csv(export_dir / "vulnerabilidades.csv", index=False)
    print(f"✅ Vulnerabilidades exportadas a: {export_dir / 'vulnerabilidades.csv'}")

if not df_codeql.empty:
    df_codeql.to_csv(export_dir / "codeql_hallazgos.csv", index=False)
    print(f"✅ Hallazgos CodeQL exportados a: {export_dir / 'codeql_hallazgos.csv'}")

print(f"\n📂 Todos los exports en: {export_dir}")

---

## 📝 Conclusiones

*(Escriba aquí sus conclusiones basadas en los resultados obtenidos)*

1. **Dependencias:** ...
2. **Vulnerabilidades encontradas:** ...
3. **Severidad predominante:** ...
4. **Paquetes más afectados:** ...
5. **Disponibilidad de correcciones:** ...
6. **Recomendaciones:** ...

---
**Curso de Ciberseguridad (ICC610) - 2026**